# Searching for Hidden Text in Music: A Complexity Sketch

In [`bach_cipher.ipynb`](bach_cipher.ipynb) we searched a MIDI file for the B-A-C-H motif — and it was easy, because we **knew** what we were looking for.

But what if we want to ask a harder question:

> *Does this piece of music contain any hidden text at all, under any cipher?*

This notebook sketches the computational problem, analyses its complexity, and explores why — even with modern computers — a naive brute-force search is completely impractical.

---

## 1. Formalising the Problem

We need to be precise about what we're searching for. Let's define the problem formally.

**Input:**
- A sequence of $N$ notes $\mathbf{m} = (m_1, m_2, \ldots, m_N)$, where each $m_i$ is a pitch class in $\{0, 1, \ldots, 11\}$
- An alphabet $\Sigma = \{A, B, \ldots, Z\}$ of size 26

**A cipher** is a function $f: \Sigma \rightarrow \{0,\ldots,11\}$ mapping each letter to a pitch class.

**Applying a cipher** to the note sequence gives a candidate text:
$$\text{decode}(\mathbf{m}, f) = (f^{-1}(m_1),\, f^{-1}(m_2),\, \ldots,\, f^{-1}(m_N))$$

**The decision problem:** Does there exist a cipher $f$ such that $\text{decode}(\mathbf{m}, f)$ contains a meaningful English word or phrase?

This is what we need to solve.

---
## 2. How Large is the Search Space?

Before writing any code, let's count how many ciphers we would need to check.

### 2a. The simplest case: any mapping, repeats allowed

Each of 26 letters maps independently to one of 12 pitch classes. There's no constraint — two letters can share a note.

In [ ]:
from math import factorial, log2, log10

PITCH_CLASSES = 12
ALPHABET_SIZE = 26

# Every letter independently picks one of 12 notes
space_unrestricted = PITCH_CLASSES ** ALPHABET_SIZE

print(f"Unrestricted mappings (12^26):")
print(f"  = {space_unrestricted:.4e}")
print(f"  ≈ 2^{log2(space_unrestricted):.1f}")
print(f"  ≈ 10^{log10(space_unrestricted):.1f}")

### 2b. A tighter model: letters grouped into 12 pitch classes

In a realistic cipher, the 26 letters are **partitioned** into 12 groups, one per pitch class. This is fewer possibilities, but still enormous.

The number of ways to partition 26 items into 12 non-empty labelled groups is the **Stirling number of the second kind** $S(26, 12)$, multiplied by $12!$ to assign groups to pitch classes.

In [ ]:
def stirling2(n, k):
    """Stirling numbers of the second kind S(n,k):
    number of ways to partition n items into k non-empty subsets."""
    if n == k:
        return 1
    if k == 0 or k > n:
        return 0
    # Recurrence: S(n,k) = k*S(n-1,k) + S(n-1,k-1)
    # Use dynamic programming
    dp = [[0] * (k + 1) for _ in range(n + 1)]
    dp[0][0] = 1
    for i in range(1, n + 1):
        for j in range(1, k + 1):
            dp[i][j] = j * dp[i-1][j] + dp[i-1][j-1]
    return dp[n][k]

s = stirling2(26, 12)
# Multiply by 12! to assign each partition to a specific pitch class
space_partitions = s * factorial(12)

print(f"Stirling S(26,12) = {s:.4e}")
print(f"Labelled partitions (S(26,12) × 12!) = {space_partitions:.4e}")
print(f"  ≈ 10^{log10(space_partitions):.1f}")
print()
print("For comparison, unrestricted (12^26) =", f"{space_unrestricted:.4e}")

### 2c. Pigeonhole principle

Note that we have 26 letters but only 12 pitch classes. By the **Pigeonhole Principle**, *at least* $26 - 12 = 14$ letters must share a note with another letter. This means the cipher is inherently **ambiguous** when decoding — one note cannot uniquely identify one letter.

Real historical ciphers dealt with this in various ways: using rhythm, dynamics, or octave as additional dimensions. We'll keep the pitch-only model for simplicity.

In [ ]:
print("Pigeonhole analysis:")
print(f"  Letters:       {ALPHABET_SIZE}")
print(f"  Pitch classes: {PITCH_CLASSES}")
print(f"  Minimum collisions (letters sharing a note): {ALPHABET_SIZE - PITCH_CLASSES}")
print()
print("Implication: decoding is one-to-many.")
print("A note maps to 26/12 ≈", round(26/12, 1), "letters on average.")

---
## 3. The Full Algorithm and Its Cost

A brute-force search has three nested stages:

```
for each cipher f in all possible ciphers:          # outer loop
    text = decode(note_sequence, f)                  # apply cipher
    if contains_english_words(text):                 # score
        report(f, text)
```

Let's cost each stage.

In [ ]:
import math

# Parameters
N = 500          # notes in a typical piece (short movement)
W = 50_000       # words in a dictionary
word_len = 5     # average word length in letters

# Cost of applying one cipher to a piece: O(N)
cost_decode = N

# Cost of scoring: slide a window of width word_len across the decoded text
# and check each substring against a dictionary.
# Naive: O(N * word_len) per check; with a hash set: O(N)
cost_score = N

# Cost per cipher
cost_per_cipher = cost_decode + cost_score

# Total operations
total_ops = space_unrestricted * cost_per_cipher

print(f"Piece length N:          {N} notes")
print(f"Cost per cipher:         O({cost_per_cipher}) operations")
print(f"Number of ciphers:       {space_unrestricted:.2e}")
print(f"Total operations:        {total_ops:.2e}")
print()

# Time on a fast modern CPU
for speed_label, ops_per_sec in [('Laptop CPU (10^9 ops/s)', 1e9),
                                   ('GPU cluster (10^15 ops/s)', 1e15),
                                   ('All CPUs on Earth (10^21 ops/s)', 1e21)]:
    seconds = total_ops / ops_per_sec
    years   = seconds / (60 * 60 * 24 * 365.25)
    print(f"  {speed_label}: {years:.1e} years")

The universe is approximately $1.4 \times 10^{10}$ years old. Even with all the computing power on Earth, brute force is **not remotely feasible** within any meaningful timescale.

---
## 4. Complexity Class

Where does this problem sit in the landscape of computational complexity?

| Property | Value |
|---|---|
| Input size | $N$ (notes) |
| Search space | $12^{26}$ — **constant** with respect to $N$ |
| Verification (given a cipher) | $O(N)$ |
| Overall | $O(12^{26} \cdot N)$ = $O(N)$ — *linear in the piece length!* |

**Wait — that looks fast.** If the cipher space is a constant, the whole thing is just $O(N)$, right?

Yes — *with respect to N*. But the constant hidden in that $O(\cdot)$ is $10^{28}$. This is the key insight:

> **Big-O notation hides constants.** When the constant is astronomical, $O(N)$ is still completely intractable.

This is different from the problems usually called "intractable". It's not NP-complete in the traditional sense — it's more like an **exhaustive key search** in cryptography, where the key space is fixed but enormous.

In [ ]:
# Visualise how runtime scales with N (piece length)
# The key point: it's linear in N, but with a terrifying constant

import matplotlib.pyplot as plt
import numpy as np

OPS_PER_SEC = 1e9   # laptop CPU
SECS_PER_YEAR = 60 * 60 * 24 * 365.25

piece_lengths = np.array([50, 100, 200, 500, 1000, 5000])
years = space_unrestricted * (piece_lengths * 2) / OPS_PER_SEC / SECS_PER_YEAR

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(piece_lengths, years, 'o-', color='steelblue', linewidth=2)
ax.axhline(1.4e10, color='firebrick', linestyle='--', label='Age of the universe')
ax.set_xlabel('Piece length (notes)')
ax.set_ylabel('Estimated runtime (years)')
ax.set_title('Brute-force cipher search: runtime vs. piece length')
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('complexity_plot.png', dpi=120)
plt.show()
print("The runtime is linear in N — but the constant factor is 10^28.")

---
## 5. Can We Do Better? Pruning the Search

The brute-force approach treats every cipher as equally plausible. In practice, we can prune the search space dramatically using **constraints**.

### Strategy 1: Known cipher families

Instead of all $12^{26}$ mappings, restrict to historically documented systems.

In [ ]:
# Known historical cipher systems (a small sample)
KNOWN_CIPHERS = {
    'German (A–H)': 8,            # only 8 letters, unique mapping
    'Solfège (Do–Si)': 7,         # Do Re Mi Fa Sol La Si → C D E F G A B
    'Alpha-numeric (A=1…)': 26,   # each letter → scale degree, cyclic
    'Chromatic (A=C…)': 26,       # A–Z mapped onto chromatic scale
}

# Each of these is essentially a single cipher or tiny family
# The search space collapses to the number of known systems

print("Restricting to known historical systems:")
print(f"  Search space: {len(KNOWN_CIPHERS)} ciphers  (down from 10^28)")
print(f"  Reduction factor: ~10^{log10(space_unrestricted / len(KNOWN_CIPHERS)):.0f}")

### Strategy 2: Frequency analysis

In English, some letters appear far more often than others. If we assume the hidden text is English, the most frequent note in the piece should map to **E** (the most common letter), the second most frequent to **T**, and so on.

This reduces the search to finding the best **permutation** of the frequency ranking — still a large space, but much smaller.

In [ ]:
from collections import Counter
import string

# English letter frequencies (from large corpus, most to least common)
ENGLISH_FREQ = 'ETAOINSHRDLCUMWFGYPBVKJXQZ'

# If we trust the frequency mapping perfectly: 1 candidate
# If we allow the top-k notes to be in any order: k! candidates

print("Search space after frequency analysis:")
print()
for k in [1, 2, 3, 4, 5, 12]:
    candidates = factorial(k) * (12 ** (26 - k))   # top-k permuted, rest free
    # More realistic: just fix top-k, fix remaining by freq order
    candidates_tight = factorial(k)
    print(f"  Top {k:2d} notes fixed by frequency → {factorial(k):>12,} candidates  "
          f"(reduction: 10^{log10(space_unrestricted/factorial(k)):.0f}x)")

### Strategy 3: Language model scoring

Rather than checking decoded text against a dictionary (which requires complete words), we can use a **character-level language model** to score how English-like a sequence of letters looks — even partial or garbled text.

A simple version: compute the **log-probability** of the decoded text under a bigram model of English. Ciphers that produce plausible letter sequences score higher.

In [ ]:
# Simple bigram language model
# We'll estimate bigram frequencies from a short sample text

import math
from collections import defaultdict

SAMPLE_TEXT = """
the quick brown fox jumps over the lazy dog
to be or not to be that is the question
all that glitters is not gold
music is the shorthand of emotion
the art of fugue by johann sebastian bach
""".upper()

def build_bigram_model(text):
    """Count bigram frequencies and return log-probabilities."""
    letters = [c for c in text if c in string.ascii_uppercase]
    counts = defaultdict(lambda: defaultdict(int))
    for a, b in zip(letters, letters[1:]):
        counts[a][b] += 1
    # Convert to log-probabilities (add-1 smoothing)
    log_probs = {}
    for a in string.ascii_uppercase:
        total = sum(counts[a].values()) + 26   # smoothing
        for b in string.ascii_uppercase:
            log_probs[(a, b)] = math.log((counts[a][b] + 1) / total)
    return log_probs

def score_text(text, log_probs):
    """Score a string using bigram log-probability (higher = more English-like)."""
    letters = [c for c in text.upper() if c in string.ascii_uppercase]
    if len(letters) < 2:
        return float('-inf')
    return sum(log_probs.get((a, b), -10) for a, b in zip(letters, letters[1:]))

log_probs = build_bigram_model(SAMPLE_TEXT)

# Compare a real English phrase vs. random letters
test_cases = [
    ("THE ART OF FUGUE",     "Real English"),
    ("BACH",                  "The cipher itself"),
    ("XKQZ",                  "Random letters"),
    ("ZZZZ",                  "All same letter"),
]

print("Bigram language model scores (higher = more English-like):")
print()
for text, label in test_cases:
    s = score_text(text, log_probs)
    print(f"  {text:25s}  {label:22s}  score: {s:.2f}")

With a language model, we can score candidates **without** needing exact word matches. This lets us search more efficiently: generate a candidate cipher, decode, score, keep only the top candidates.

This is essentially the approach used in **automatic cryptanalysis** — solving classical ciphers computationally.

---
## 6. A Tractable Heuristic: Hill Climbing

Rather than enumerating all ciphers, we can use a **hill-climbing** search:

1. Start with a random cipher
2. Decode the piece and score the result
3. Try swapping two letter assignments
4. If the score improves, keep the swap
5. Repeat until no swap improves the score

This is a **local search** heuristic — it won't find the global optimum, but it's fast and often finds good solutions.

In [ ]:
import random

def random_cipher():
    """Generate a random mapping from each letter to a pitch class (0-11)."""
    return {letter: random.randint(0, 11) for letter in string.ascii_uppercase}

def apply_cipher(note_sequence, cipher):
    """Decode a note sequence using a cipher (note -> letter).
    Since multiple letters can share a note, return the most common one."""
    # Invert: pitch class -> list of letters
    pc_to_letters = defaultdict(list)
    for letter, pc in cipher.items():
        pc_to_letters[pc].append(letter)
    # For each note, pick the first matching letter (simplified)
    result = []
    for note in note_sequence:
        pc = note % 12
        letters = pc_to_letters.get(pc, ['?'])
        result.append(letters[0])
    return ''.join(result)

def hill_climb(note_sequence, log_probs, iterations=500, restarts=3):
    """Hill-climbing search for the best cipher."""
    best_cipher = None
    best_score  = float('-inf')

    for restart in range(restarts):
        cipher = random_cipher()
        score  = score_text(apply_cipher(note_sequence, cipher), log_probs)

        for _ in range(iterations):
            # Try swapping the pitch assignments of two random letters
            a, b = random.sample(list(string.ascii_uppercase), 2)
            new_cipher = dict(cipher)
            new_cipher[a], new_cipher[b] = new_cipher[b], new_cipher[a]
            new_score = score_text(apply_cipher(note_sequence, new_cipher), log_probs)
            if new_score > score:
                cipher, score = new_cipher, new_score

        if score > best_score:
            best_score, best_cipher = score, cipher

    return best_cipher, best_score

# --- Demo ---
# Create a test: encode a short phrase, then try to recover it
SECRET = "BACH"

# Build a simple cipher to encode it
NOTE_NAMES = {0:'C',1:'C#',2:'D',3:'Eb',4:'E',5:'F',
              6:'F#',7:'G',8:'Ab',9:'A',10:'Bb',11:'B'}
GERMAN_CIPHER = {'A':69,'B':70,'C':60,'D':62,'E':64,'F':65,'G':67,'H':71}
secret_notes = [GERMAN_CIPHER[c] for c in SECRET]

print(f"Secret phrase: '{SECRET}'")
print(f"Encoded notes: {[NOTE_NAMES[n % 12] for n in secret_notes]}")
print()
print("Running hill-climbing search...")

random.seed(42)
best_cipher, best_score = hill_climb(secret_notes, log_probs, iterations=300, restarts=5)
decoded = apply_cipher(secret_notes, best_cipher)

print(f"Best decoded text: '{decoded}'  (score: {best_score:.2f})")
print()
print("Note: with only 4 notes and a tiny language model, exact recovery")
print("is unlikely — but the score gives a principled way to rank candidates.")

---
## 7. Summary: The Complexity Landscape

| Approach | Search space | Feasible? |
|---|---|---|
| Brute force (all mappings) | $12^{26} \approx 10^{28}$ | ✗ Never |
| Brute force (partitions) | $S(26,12) \times 12! \approx 10^{24}$ | ✗ Never |
| Known cipher families | $\sim 10$ systems | ✓ Instant |
| Frequency-guided search | $12! \approx 5 \times 10^{8}$ | ✓ Minutes |
| Hill climbing (heuristic) | $O(k \cdot 26^2)$ per restart | ✓ Seconds |

### Key lessons

1. **The problem is not NP-hard in the formal sense** — verifying a solution (applying one cipher and checking the result) is $O(N)$, fast.

2. **The difficulty is the search space**, which is exponential in the alphabet size — not in the problem input. This is sometimes called **combinatorial explosion**.

3. **Domain knowledge transforms the problem.** The German cipher was found because scholars knew what to look for. The computational challenge is what to do without that knowledge.

4. **Language models are the key tool.** If we can score how English-like a decoded string is, we can use gradient-based or heuristic search rather than exhaustive enumeration.

5. **This pattern recurs across the digital humanities** — from authorship attribution to stylometry to manuscript stemma reconstruction. The hard part is rarely verification; it's defining and searching a space of hypotheses.

---
*Humanities Programming class — University of Oxford*